# Experimento GRPO inglés → ruso — notebook autocontenido

Este archivo contiene en sus propias celdas las funciones de carga de datos, LoRA, generación, recompensa chrF++, pérdida GRPO, entrenamiento, evaluación y presentación de resultados. **No importa código Python del repositorio ni llama a `main.py`, `utils.py`, `dl.py` o `collect_results.py`.**

Para ejecutarlo fuera del repositorio solo hacen falta Python, las dependencias indicadas abajo, conexión a Hugging Face para obtener modelo/dataset y, si se quiere una ejecución rápida, una GPU con memoria suficiente. El límite de duración se controla entre pasos de entrenamiento; una evaluación o una generación que ya esté en curso no se puede interrumpir con seguridad a mitad del lote.

Los resultados históricos incluidos son los obtenidos anteriormente en este proyecto, no resultados de una ejecución de este notebook. Las muestras de 4/32 ejemplos solo validan el flujo; no son métricas representativas.

## Resultados previos registrados

Esta tabla reproduce el formato de `collect_results.py` para los runs que ya se ejecutaron. `base` es NLLB antes del entrenamiento y `ours` es el chrF++ de test al final. No hay resultados de los métodos BT/UMT/RT en estos experimentos pequeños.

| Configuración | Ejemplos train | Épocas | Ejemplos test | Base chrF++ | Ours chrF++ | Duración observada |
|---|---:|---:|---:|---:|---:|---:|
| Smoke test | 4 | 1 | 4 | 32.30 | 32.30 | ~1 min 25 s |
| Run ampliado | 16 | 1 | 32 | 41.25 | 41.25 | ~4 min 20 s |
| Run ampliado | 16 | 2 | 32 | 41.25 | 41.25 | ~9 min 27 s |

Las dos últimas métricas se calculan sobre los mismos primeros 32 ejemplos de test; no se observa mejora en esa muestra. La prueba de 32 ejemplos train × 2 épocas se canceló antes de acabar y no se incluye como resultado completo.

## 1. Dependencias

Entorno de referencia: Python 3.11–3.12, `torch`, `transformers`, `datasets<4`, `sacrebleu` y `tqdm`. `datasets==3.6.0` se usa porque el dataset de esta tarea se carga con `trust_remote_code=True`. Instala las dependencias en el entorno del kernel antes de ejecutar las siguientes celdas, por ejemplo:

```text
%pip install torch transformers==4.57.1 datasets==3.6.0 sacrebleu==2.5.1 tqdm
```

La instalación de PyTorch adecuada depende de tu plataforma y soporte CUDA; consulta el selector oficial de PyTorch si necesitas GPU. No se instala ni descarga nada automáticamente desde este notebook.

In [10]:
import copy
import json
import math
import random
import time
from datetime import datetime
from pathlib import Path
from typing import Sequence

import torch
from datasets import load_dataset
from sacrebleu.metrics import BLEU, CHRF
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print('torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

torch: 2.8.0+cu128
CUDA disponible: True
GPU: NVIDIA GeForce RTX 3070


## 2. Ajustes

El preset permite repetir el último experimento completo: LoRA, 16 ejemplos de entrenamiento, 2 épocas y hasta 32 ejemplos de evaluación/test. El entrenamiento no comienza hasta cambiar `RUN_TRAINING` a `True`. Cada ejecución obtiene una carpeta nueva en `standalone_results/` bajo el directorio actual.

`MAX_RUNTIME_SECONDS` limita el entrenamiento entre lotes. Se reserva tiempo para guardar las evaluaciones finales; aun así, un lote largo o la descarga inicial del modelo/dataset puede sobrepasar el tiempo esperado. Para probar rápidamente, baja train/test a 4–8 ejemplos y una época.

In [11]:
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
DATASET_NAME = 'breakend/nllb-multi-domain'
SOURCE_LANG = 'eng_Latn'
TARGET_LANG = 'rus_Cyrl'
DATASET_CONFIG = f'{SOURCE_LANG}-{TARGET_LANG}'

USE_LORA = True
LORA_RANK = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ('q_proj', 'v_proj')

MAX_TRAIN_SAMPLES = 16  # None usa todo el split train
MAX_EVAL_SAMPLES = 32   # None usa todo valid y test
EPOCHS = 2
BATCH_SIZE = 2
NUM_RETURN_SEQUENCES = 4
MAX_NEW_TOKENS = 63
LEARNING_RATE = 2e-6
BETA = 0.04
CLIP_PARAM = 0.2
GEN_TEMPERATURE = 1.8
TOP_K = 100
TOP_P = 0.95
EVAL_EVERY_N_OPT_STEPS = 0  # 0 desactiva evaluación intermedia
RUN_VALIDATION = True
RUN_TEST = True
SAVE_ADAPTER = False
SEED = 27
MAX_RUNTIME_SECONDS = 14 * 60
RUN_TRAINING = True  # cambiar a True solo cuando se quiera iniciar el experimento

if EPOCHS < 1 or BATCH_SIZE < 1 or NUM_RETURN_SEQUENCES < 2:
    raise ValueError('Épocas/batch positivos y al menos dos secuencias para calcular ventajas GRPO.')
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < 1:
    raise ValueError('MAX_TRAIN_SAMPLES debe ser None o positivo.')
if MAX_EVAL_SAMPLES is not None and MAX_EVAL_SAMPLES < 1:
    raise ValueError('MAX_EVAL_SAMPLES debe ser None o positivo.')
if EVAL_EVERY_N_OPT_STEPS < 0:
    raise ValueError('EVAL_EVERY_N_OPT_STEPS no puede ser negativo.')

print(f'{MODEL_NAME} | {SOURCE_LANG} → {TARGET_LANG} | LoRA={USE_LORA}')
print(f'train={MAX_TRAIN_SAMPLES}, eval/test={MAX_EVAL_SAMPLES}, épocas={EPOCHS}, batch={BATCH_SIZE}')

facebook/nllb-200-distilled-600M | eng_Latn → rus_Cyrl | LoRA=True
train=16, eval/test=32, épocas=2, batch=2


Fijamos semillas para reproducibilidad aproximada. La generación del entrenamiento es estocástica; el mismo seed no garantiza bit a bit el mismo resultado en todas las GPU/versiones de CUDA.

In [12]:
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = Path.cwd() / 'standalone_results' / datetime.now().strftime('run_%Y%m%d_%H%M%S')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
METRICS_PATH = OUTPUT_DIR / 'metrics.jsonl'
print('Dispositivo:', DEVICE)
print('Resultados nuevos:', OUTPUT_DIR.resolve())

Dispositivo: cuda
Resultados nuevos: /home/miguel/MT-via-Round-Trip-RL/notebooks/standalone_results/run_20261021_182719


## 3. Adaptadores LoRA y carga del modelo

La función reemplaza módulos lineales cuyo nombre termina en `q_proj` o `v_proj` por una capa base congelada más una rama de bajo rango entrenable. La inicialización de la rama B a cero mantiene inicialmente la salida de la capa original. Si LoRA está desactivado, los pesos del modelo completo permanecen entrenables.

In [13]:
class LoRALinear(torch.nn.Module):
    def __init__(self, base_layer: torch.nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        if r < 1:
            raise ValueError('LoRA rank must be positive.')
        if not 0.0 <= dropout < 1.0:
            raise ValueError('LoRA dropout must be in [0, 1).')
        self.base_layer = base_layer
        self.base_layer.requires_grad_(False)
        self.lora_A = torch.nn.Linear(base_layer.in_features, r, bias=False,
                                      device=base_layer.weight.device, dtype=base_layer.weight.dtype)
        self.lora_B = torch.nn.Linear(r, base_layer.out_features, bias=False,
                                      device=base_layer.weight.device, dtype=base_layer.weight.dtype)
        self.dropout = torch.nn.Dropout(dropout)
        self.scaling = alpha / r
        torch.nn.init.kaiming_uniform_(self.lora_A.weight, a=5**0.5)
        torch.nn.init.zeros_(self.lora_B.weight)

    def forward(self, inputs):
        return self.base_layer(inputs) + self.lora_B(self.lora_A(self.dropout(inputs))) * self.scaling


def apply_lora(model, target_modules, r, alpha, dropout):
    targets = [(name, module) for name, module in model.named_modules()
               if name.rsplit('.', 1)[-1] in target_modules]
    if not targets:
        raise ValueError(f'No modules matched LoRA targets {target_modules}.')
    replaced = []
    for name, module in targets:
        if not isinstance(module, torch.nn.Linear):
            raise TypeError(f'{name} is not a torch.nn.Linear layer.')
        parent_name, _, child_name = name.rpartition('.')
        parent = model.get_submodule(parent_name) if parent_name else model
        setattr(parent, child_name, LoRALinear(module, r, alpha, dropout))
        replaced.append(name)
    return replaced


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if USE_LORA:
    model.requires_grad_(False)
    adapted_layers = apply_lora(model, LORA_TARGETS, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
    print(f'LoRA: {len(adapted_layers)} capas adaptadas, rank={LORA_RANK}')
model.to(DEVICE)
if hasattr(tokenizer, 'src_lang'):
    tokenizer.src_lang = SOURCE_LANG
if hasattr(tokenizer, 'tgt_lang'):
    tokenizer.tgt_lang = TARGET_LANG
trainable = [p for p in model.parameters() if p.requires_grad]
trainable_count = sum(p.numel() for p in trainable)
total_count = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables: {trainable_count:,} ({trainable_count / total_count:.2%}) / {total_count:,}')

LoRA: 72 capas adaptadas, rank=8
Parámetros entrenables: 1,179,648 (0.19%) / 616,253,440


## 4. Dataset y dataloaders

Se descargan los splits del dataset público y se aplican los límites seleccionados. Para mantener el mismo comportamiento que el experimento del repositorio, validación y test usan batch size 48; los límites pequeños suelen formar un solo lote de evaluación.

In [14]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, trust_remote_code=True)
print('Tamaños originales:', {split: len(rows) for split, rows in dataset.items()})

def find_split(dataset_dict, names, limit=None, required=False):
    for name in names:
        if name in dataset_dict:
            split = dataset_dict[name]
            if limit is not None:
                split = split.select(range(min(int(limit), len(split))))
            return split
    if required:
        raise ValueError(f'Required dataset split not found: {names}')
    return None

class TranslationDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
        self.source_column = f'sentence_{SOURCE_LANG}'
        self.target_column = f'sentence_{TARGET_LANG}'

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        source = row[self.source_column]
        target = row[self.target_column]
        return source, target, source, index

def collate_translation(batch):
    sources, targets, original_sources, sample_ids = zip(*batch)
    encoded = tokenizer(list(sources), padding=True, truncation=True, return_tensors='pt')
    return encoded, list(targets), list(original_sources), list(sample_ids)

train_rows = find_split(dataset, ('train', 'training'), MAX_TRAIN_SAMPLES, required=True)
valid_rows = find_split(dataset, ('valid', 'validation', 'val'), MAX_EVAL_SAMPLES, required=RUN_VALIDATION)
test_rows = find_split(dataset, ('test', 'test_final'), MAX_EVAL_SAMPLES, required=RUN_TEST)
train_data = TranslationDataset(train_rows)
valid_data = TranslationDataset(valid_rows) if valid_rows is not None else None
test_data = TranslationDataset(test_rows) if test_rows is not None else None
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_translation)
valid_loader = DataLoader(valid_data, batch_size=48, shuffle=False, collate_fn=collate_translation) if valid_data else None
test_loader = DataLoader(test_data, batch_size=48, shuffle=False, collate_fn=collate_translation) if test_data else None
print(f'Train: {len(train_data)} ejemplos / {len(train_loader)} lotes por época')
print(f'Validación: {len(valid_data) if valid_data else 0} ejemplos; test: {len(test_data) if test_data else 0} ejemplos')

Tamaños originales: {'train': 6000, 'valid': 1309, 'test': 1500}
Train: 16 ejemplos / 8 lotes por época
Validación: 32 ejemplos; test: 32 ejemplos


## 5. Evaluación y guardado de métricas

La evaluación es determinista (`do_sample=False`, un beam) y calcula corpus chrF++ como en el proyecto. Se guarda cada resultado en `metrics.jsonl`; la primera evaluación es el modelo base y la última es la evaluación posterior al entrenamiento.

In [15]:
def append_metric(split_name, step, score):
    record = {'split': split_name, 'step': int(step), f'{split_name}/chrf++': float(score)}
    with METRICS_PATH.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

def run_evaluation(model, tokenizer, loader, target_token_id, split_name, step):
    if loader is None:
        return None
    was_training = model.training
    model.eval()
    predictions, references = [], []
    with torch.no_grad():
        for encoder_inputs, ground_truths, source_texts, sample_ids in tqdm(loader, desc=f'Evaluating ({split_name})'):
            encoder_inputs = {key: value.to(DEVICE, non_blocking=True) for key, value in encoder_inputs.items()}
            generated = model.generate(
                input_ids=encoder_inputs['input_ids'],
                attention_mask=encoder_inputs.get('attention_mask'),
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=1,
                do_sample=False,
                forced_bos_token_id=target_token_id,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
            references.extend(str(reference) for reference in ground_truths)
    score = CHRF(word_order=2, char_order=6).corpus_score(predictions, [references]).score
    print(f'{split_name} chrF++: {score:.4f}')
    append_metric(split_name, step, score)
    if was_training:
        model.train()
    return float(score)

target_lang_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)
source_lang_id = tokenizer.convert_tokens_to_ids(SOURCE_LANG)

def evaluate_enabled_splits(step):
    scores = {}
    if RUN_TEST:
        scores['test'] = run_evaluation(model, tokenizer, test_loader, target_lang_id, 'test', step)
    if RUN_VALIDATION:
        scores['eval'] = run_evaluation(model, tokenizer, valid_loader, target_lang_id, 'eval', step)
    return scores

## 6. Generación de candidatos y objetivo GRPO

Para cada lote se generan cuatro traducciones candidatas; cada traducción se vuelve a traducir a inglés. La recompensa es chrF normalizado a [0, 1], estandarizado dentro del grupo de candidatos. La pérdida incluye un término tipo PPO y una penalización KL contra una copia congelada de la política de referencia.

Las siguientes funciones están transcritas aquí para que el notebook no dependa de `utils.py`.

In [16]:
@torch.no_grad()
def grpo_generate_sequences(model, tokenizer, encoder_inputs, target_id):
    return model.generate(
        input_ids=encoder_inputs['input_ids'],
        attention_mask=encoder_inputs.get('attention_mask'),
        forced_bos_token_id=target_id,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=GEN_TEMPERATURE,
        num_return_sequences=NUM_RETURN_SEQUENCES,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        top_k=TOP_K,
        top_p=TOP_P,
    )

def gather_token_log_probs(logits, target_ids):
    return torch.gather(logits.log_softmax(dim=-1), -1, target_ids.unsqueeze(-1)).squeeze(-1)

def decoder_per_token_logps(policy, tokenizer, encoder_inputs, decoder_input_ids, target_ids):
    device = next(policy.parameters()).device
    base_batch = encoder_inputs['input_ids'].size(0)
    if decoder_input_ids.size(0) % base_batch:
        raise ValueError('Decoder batch must be a multiple of encoder batch.')
    multiplier = decoder_input_ids.size(0) // base_batch
    input_ids = encoder_inputs['input_ids'].repeat_interleave(multiplier, dim=0).to(device)
    attention_mask = encoder_inputs.get('attention_mask')
    if attention_mask is not None:
        attention_mask = attention_mask.repeat_interleave(multiplier, dim=0).to(device)
    decoder_input_ids = decoder_input_ids.to(device)
    target_ids = target_ids.to(device)
    decoder_mask = (decoder_input_ids != tokenizer.pad_token_id).long()
    outputs = policy(input_ids=input_ids, attention_mask=attention_mask,
                     decoder_input_ids=decoder_input_ids,
                     decoder_attention_mask=decoder_mask, use_cache=False)
    return gather_token_log_probs(outputs.logits, target_ids)

def grpo_compute_loss_and_logs(policy, reference_policy, tokenizer, encoder_inputs,
                               generated_sequences, ground_truths, target_lang_id):
    if generated_sequences.ndim != 3:
        raise ValueError('Expected candidates with shape (batch, candidates, sequence).')
    device = next(policy.parameters()).device
    batch_size, candidate_count, sequence_length = generated_sequences.shape
    if batch_size != len(ground_truths):
        raise ValueError('Ground-truth count does not match generated batch size.')
    flat = generated_sequences.to(device).reshape(batch_size * candidate_count, sequence_length)
    decoder_inputs, target_ids = flat[:, :-1], flat[:, 1:]
    logps = decoder_per_token_logps(policy, tokenizer, encoder_inputs, decoder_inputs, target_ids)
    with torch.no_grad():
        reference_logps = decoder_per_token_logps(reference_policy, tokenizer, encoder_inputs,
                                                  decoder_inputs, target_ids).to(device)

    is_pad = target_ids == tokenizer.pad_token_id
    is_eos = target_ids == tokenizer.eos_token_id
    is_lang = target_ids == target_lang_id
    completion_mask = (~is_pad) & (~is_eos.cumsum(dim=-1).ge(1)) & (~is_lang)
    completion_mask = completion_mask.reshape(batch_size, candidate_count, -1)
    decoded = tokenizer.batch_decode(
        torch.where(flat == tokenizer.pad_token_id, tokenizer.eos_token_id, flat).cpu(),
        skip_special_tokens=True,
    )
    references = [ground_truths[i // candidate_count] for i in range(len(decoded))]
    chrf_scores, bleu_scores = [], []
    for hypothesis, reference in zip(decoded, references):
        chrf_scores.append(CHRF(word_order=2, char_order=6).corpus_score([hypothesis], [[reference]]).score / 100)
        bleu_scores.append(BLEU(effective_order=True).corpus_score([hypothesis], [[reference]]).score / 100)
    chrf_tensor = torch.tensor(chrf_scores, device=device)
    bleu_tensor = torch.tensor(bleu_scores, device=device)
    rewards = chrf_tensor.reshape(batch_size, candidate_count)
    advantages = ((rewards - rewards.mean(dim=1, keepdim=True)) /
                  (rewards.std(dim=1, keepdim=True) + 1e-4)).unsqueeze(-1)

    logps = logps.reshape(batch_size, candidate_count, -1)
    advantages = advantages.expand_as(logps)
    ratio = torch.exp(logps - logps.detach())
    clipped_ratio = torch.clamp(ratio, 1 - CLIP_PARAM, 1 + CLIP_PARAM)
    policy_objective = torch.minimum(ratio * advantages, clipped_ratio * advantages)
    ref_minus_policy = reference_logps.reshape(batch_size, candidate_count, -1) - logps
    kl = torch.exp(ref_minus_policy) - ref_minus_policy - 1.0
    objective = (policy_objective - BETA * kl) * completion_mask
    token_counts = completion_mask.sum(dim=-1).clamp_min(1)
    loss = (-objective.sum(dim=-1) / token_counts).mean()
    with torch.no_grad():
        kl_mean = (kl * completion_mask).sum() / token_counts.sum()
    logs = {
        'loss': loss.detach(), 'kl': kl_mean.detach(), 'reward': rewards.mean().detach(),
        'chrf': chrf_tensor.mean().detach(), 'bleu': bleu_tensor.mean().detach(),
    }
    return loss, logs

## 7. Entrenamiento completo

Se evalúa test/validación antes de entrenar, se ejecuta una actualización por lote y se vuelven a evaluar los splits activados al final. La política de referencia se refresca cada 16 actualizaciones, siguiendo el comportamiento del script original. Las evaluaciones intermedias están desactivadas en el preset (`EVAL_EVERY_N_OPT_STEPS=0`).

In [17]:
def train_experiment():
    if not RUN_TRAINING:
        print('Ejecución omitida: cambia RUN_TRAINING=True para comenzar.')
        return None
    torch.manual_seed(SEED)
    model.train()
    reference_policy = copy.deepcopy(model).to(DEVICE)
    reference_policy.eval()
    reference_policy.requires_grad_(False)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
    optimizer.zero_grad()
    print(f'Entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
    started = time.monotonic()
    total_steps = EPOCHS * len(train_loader)
    initial_scores = evaluate_enabled_splits(step=0)
    optimizer_step = 0
    completed = True

    for epoch in range(EPOCHS):
        step_idx = 0
        for batch in train_loader:
            if time.monotonic() - started >= MAX_RUNTIME_SECONDS:
                print('Límite de duración alcanzado; se omite el resto del entrenamiento y la evaluación final.')
                completed = False
                break
            encoder_inputs, ground_truths, source_texts, sample_ids = batch
            encoder_inputs = {key: value.to(DEVICE, non_blocking=True) for key, value in encoder_inputs.items()}
            batch_size = encoder_inputs['input_ids'].size(0)
            forward_generated = grpo_generate_sequences(model, tokenizer, encoder_inputs, target_lang_id)
            sequence_length = forward_generated.size(1)
            forward_grouped = forward_generated.reshape(batch_size, NUM_RETURN_SEQUENCES, sequence_length)
            forward_texts = tokenizer.batch_decode(forward_grouped.reshape(-1, sequence_length), skip_special_tokens=True)
            repeated_references = [ground_truths[i // NUM_RETURN_SEQUENCES] for i in range(len(forward_texts))]
            forward_chrf = sum(CHRF(word_order=2, char_order=6).corpus_score([hyp], [[ref]]).score
                               for hyp, ref in zip(forward_texts, repeated_references)) / len(forward_texts)

            previous_source_lang = getattr(tokenizer, 'src_lang', None)
            if previous_source_lang is not None:
                tokenizer.src_lang = TARGET_LANG
            backward_encoded = tokenizer(forward_texts, padding=True, truncation=True, return_tensors='pt')
            if previous_source_lang is not None:
                tokenizer.src_lang = previous_source_lang
            backward_inputs = {key: value.to(DEVICE, non_blocking=True) for key, value in backward_encoded.items()}
            backward_batch_size = backward_inputs['input_ids'].size(0)
            backward_generated = grpo_generate_sequences(model, tokenizer, backward_inputs, source_lang_id)
            backward_grouped = backward_generated.reshape(backward_batch_size, NUM_RETURN_SEQUENCES,
                                                          backward_generated.size(1))
            expanded_sources = [source_texts[i // NUM_RETURN_SEQUENCES] for i in range(backward_batch_size)]
            loss, logs = grpo_compute_loss_and_logs(
                model, reference_policy, tokenizer, backward_inputs, backward_grouped,
                expanded_sources, source_lang_id,
            )
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            optimizer_step += 1

            if optimizer_step % 16 == 0:
                reference_policy = copy.deepcopy(model).to(DEVICE)
                reference_policy.eval()
                reference_policy.requires_grad_(False)
            if RUN_VALIDATION and EVAL_EVERY_N_OPT_STEPS and optimizer_step % EVAL_EVERY_N_OPT_STEPS == 0:
                run_evaluation(model, tokenizer, valid_loader, target_lang_id, 'eval', step_idx)

            print(f'[epoch {epoch}] step {step_idx} (opt {optimizer_step}) | '
                  f'f_chrf={forward_chrf:.4f} | b_loss={logs["loss"].item():.4f} '
                  f'b_kl={logs["kl"].item():.4f} b_reward={logs["reward"].item():.4f} '
                  f'b_chrf={logs["chrf"].item():.4f} b_bleu={logs["bleu"].item():.4f}')
            print(f'Source[{sample_ids[0]}]: {source_texts[0]}')
            print(f'Reference[{sample_ids[0]}]: {ground_truths[0]}')
            print(f'Generated[{sample_ids[0]}]: {tokenizer.decode(forward_grouped[0, 0], skip_special_tokens=True)}')
            print(f'Back-Generated[{sample_ids[0]}]: {tokenizer.decode(backward_grouped[0, 0], skip_special_tokens=True)}')
            step_idx += 1
        if not completed:
            break

    final_scores = {}
    if completed:
        if RUN_VALIDATION:
            final_scores['eval'] = run_evaluation(model, tokenizer, valid_loader, target_lang_id, 'eval', step_idx)
        if RUN_TEST:
            final_scores['test'] = run_evaluation(model, tokenizer, test_loader, target_lang_id, 'test', step_idx)
        if SAVE_ADAPTER:
            model.save_pretrained(OUTPUT_DIR / 'model')
            tokenizer.save_pretrained(OUTPUT_DIR / 'model')
    elapsed = time.monotonic() - started
    print(f'Duración: {elapsed / 60:.2f} min | pasos completados: {optimizer_step}/{total_steps}')
    return {'initial': initial_scores, 'final': final_scores, 'steps': optimizer_step,
            'completed': completed, 'elapsed_seconds': elapsed}

run_summary = train_experiment()

Entrenables: 1,179,648


Evaluating (test): 100%|██████████| 1/1 [00:02<00:00,  2.80s/it]


test chrF++: 41.2490


Evaluating (eval): 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]


eval chrF++: 31.4218
[epoch 0] step 0 (opt 1) | f_chrf=15.8999 | b_loss=0.0128 b_kl=0.3075 b_reward=0.1548 b_chrf=0.1548 b_bleu=0.0138
Source[14]: The new car has been wonderful! It's got such a smooth but powerful ride!  Have you ever driven a BMW?  If not, you need to experience it sometime.  They've got a heavenly road feel.
Reference[14]: Новая машина замечательна! Она едет так ровно, но одновременно мощно!  Ты когда-нибудь водил BMW?  Если нет, тебе нужно как-нибудь попробовать.  Они дают просто неземное чувство дороги.
Generated[14]: Воздаст вам хорошее отношение к его команде! (вик и все они его больше?, вот что Вы понятны.) (далее - вопрос: Достойные модели?
Back-Generated[14]: Well it will give you to everyone in that business really well to the team! (Call they the big guys back and all, we get another... that will change for sure
[epoch 0] step 1 (opt 2) | f_chrf=13.1241 | b_loss=0.0196 b_kl=0.4145 b_reward=0.1471 b_chrf=0.1471 b_bleu=0.0124
Source[8]: Our set is usually an 

Evaluating (eval): 100%|██████████| 1/1 [00:09<00:00,  9.94s/it]


eval chrF++: 31.4218


Evaluating (test): 100%|██████████| 1/1 [00:13<00:00, 13.03s/it]

test chrF++: 41.2490
Duración: 5.96 min | pasos completados: 16/16


## 8. Tabla de resultados

Esta celda implementa localmente el formato de la tabla de `collect_results.py`; no importa ni ejecuta ese script. Para el run actual lee la primera y la última línea `test/chrf++` de `metrics.jsonl`. También muestra las mediciones históricas embebidas arriba para comparar.

In [18]:
def read_test_scores(metrics_path):
    if not metrics_path.exists():
        return None, None
    records = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    test_scores = [record['test/chrf++'] for record in records if record.get('split') == 'test']
    if not test_scores:
        return None, None
    return test_scores[0], test_scores[-1] if len(test_scores) > 1 else None

base_score, final_score = read_test_scores(METRICS_PATH)
print('\n### NLLB-distilled-600M (test chrF++, eng -> rus_Cyrl)\n')
print('| lang | base | bt | umt | rt | ours |')
print('|---|---:|---:|---:|---:|---:|')
base_text = '-' if base_score is None else f'{base_score:.2f}'
final_text = '-' if final_score is None else f'{final_score:.2f}'
print(f'| {TARGET_LANG} | {base_text} | - | - | - | {final_text} |')
print(f'\nMétricas detalladas: {METRICS_PATH.resolve()}')


### NLLB-distilled-600M (test chrF++, eng -> rus_Cyrl)

| lang | base | bt | umt | rt | ours |
|---|---:|---:|---:|---:|---:|
| rus_Cyrl | 41.25 | - | - | - | 41.25 |

Métricas detalladas: /home/miguel/MT-via-Round-Trip-RL/notebooks/standalone_results/run_20261021_182719/metrics.jsonl


## Interpretación y límites

- Un valor final igual al inicial significa que **esta muestra y esta configuración no muestran una mejora medible**, no que el método sea imposible de mejorar.
- Los 32 ejemplos evaluados no permiten conclusiones fiables de calidad. Para una evaluación válida, usa más datos de test y no los uses para ajustar hiperparámetros repetidamente.
- LoRA actualiza solo adaptadores, pero la generación forward/backward y la evaluación siguen computando con el modelo base completo.
- El objetivo está escrito de forma transparente para poder estudiarlo. Si se modifica su formulación, ya no será una réplica exacta del algoritmo de referencia.
- El directorio `standalone_results/` y el dataset/modelo descargados son los únicos recursos externos generados; el resto del código está en este notebook.